In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

Data Loading and Inspection:
The stock price dataset is loaded and inspected to verify its structure, column names, and missing values. This step ensures that the data is clean and suitable for supervised learning.

The dataset contains daily stock prices for several companies, and AAPL will be used as the prediction target in the regression task.

In [3]:
df = pd.read_csv('../data/raw/Prices.csv')
df.head()

,Date,AAPL,AMZN,GOOGL,JNJ,JPM,META,MSFT,NVDA,PG,XOM
0,2015-01-02,24.237549,15.4260,26.278944,76.955551,46.511124,77.905800,39.858452,0.483011,66.518349,57.916889
1,2015-01-05,23.554735,15.1095,25.778227,76.418083,45.067196,76.654541,39.491920,0.474853,66.202065,56.332203
2,2015-01-06,23.556952,14.7645,25.142035,76.042580,43.898651,75.621750,38.912289,0.460457,65.900513,56.032726
3,2015-01-07,23.887281,14.9210,25.068092,77.721291,43.965630,75.621750,39.406689,0.459257,66.246216,56.600468
4,2015-01-08,24.805077,15.0230,25.155436,78.332405,44.948116,77.637688,40.565952,0.476533,67.003761,57.542580


In [4]:
print("Shape of dataset:", df.shape)
print("\nColumn names:")
print(df.columns)

print("\nMissing values:")
print(df.isnull().sum())

Shape of dataset: (2765, 11)

Column names:
Index(['Date', 'AAPL', 'AMZN', 'GOOGL', 'JNJ', 'JPM', 'META', 'MSFT', 'NVDA',
       'PG', 'XOM'],
      dtype='object')

Missing values:
Date     0
AAPL     0
AMZN     0
GOOGL    0
JNJ      0
JPM      0
META     0
MSFT     0
NVDA     0
PG       0
XOM      0
dtype: int64


In [5]:
df["Date"] = pd.to_datetime(df["Date"])
df.head()

,Date,AAPL,AMZN,GOOGL,JNJ,JPM,META,MSFT,NVDA,PG,XOM
0,2015-01-02,24.237549,15.4260,26.278944,76.955551,46.511124,77.905800,39.858452,0.483011,66.518349,57.916889
1,2015-01-05,23.554735,15.1095,25.778227,76.418083,45.067196,76.654541,39.491920,0.474853,66.202065,56.332203
2,2015-01-06,23.556952,14.7645,25.142035,76.042580,43.898651,75.621750,38.912289,0.460457,65.900513,56.032726
3,2015-01-07,23.887281,14.9210,25.068092,77.721291,43.965630,75.621750,39.406689,0.459257,66.246216,56.600468
4,2015-01-08,24.805077,15.0230,25.155436,78.332405,44.948116,77.637688,40.565952,0.476533,67.003761,57.542580


Data Transformation:
To improve model performance, stock prices are converted into daily returns using percentage change. This transformation removes trends and allows the model to learn relationships between relative changes rather than absolute price levels.

This is important because stock prices are non-stationary and can lead to poor regression performance if used directly.

In [9]:
df_returns = df.copy()

# Convert to returns
df_returns.iloc[:, 1:] = df_returns.iloc[:, 1:].pct_change()

# Drop first row (NaN from pct_change)
df_returns = df_returns.dropna()

df_returns.head()

,Date,AAPL,AMZN,GOOGL,JNJ,JPM,META,MSFT,NVDA,PG,XOM
1,2015-01-05,-0.028172,-0.020517,-0.019054,-0.006984,-0.031045,-0.016061,-0.009196,-0.016890,-0.004755,-0.027361
2,2015-01-06,0.000094,-0.022833,-0.024679,-0.004914,-0.025929,-0.013473,-0.014677,-0.030318,-0.004555,-0.005316
3,2015-01-07,0.014023,0.010600,-0.002941,0.022076,0.001526,0.000000,0.012705,-0.002606,0.005246,0.010132
4,2015-01-08,0.038422,0.006836,0.003484,0.007863,0.022347,0.026658,0.029418,0.037617,0.011435,0.016645
5,2015-01-09,0.001073,-0.011749,-0.012211,-0.013629,-0.017387,-0.005628,-0.008405,0.004028,-0.009330,-0.001410


In [11]:
X = df_returns.drop(columns=["Date", "AAPL"])
y = df_returns["AAPL"]

In [6]:
X = df.drop(columns=["Date", "AAPL"])
y = df["AAPL"]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("\nFeature columns:")
print(X.columns)

X shape: (2765, 9)
y shape: (2765,)

Feature columns:
Index(['AMZN', 'GOOGL', 'JNJ', 'JPM', 'META', 'MSFT', 'NVDA', 'PG', 'XOM'], dtype='object')


Train-Test Split:
The dataset is divided into training and testing sets using an 80/20 split. Because the data is time-ordered stock price data, the split is done sequentially instead of randomly.

This preserves temporal order and avoids data leakage from future observations into the training set.

In [13]:
split_index = int(len(df_returns) * 0.8)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

In [14]:
split_index = int(len(df) * 0.8)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

print("Training set size:", X_train.shape)
print("Testing set size:", X_test.shape)

Training set size: (2212, 9)
Testing set size: (552, 9)


In [10]:
X = df_returns.drop(columns=["Date", "AAPL"])
y = df_returns["AAPL"]

Linear Regression:
Linear Regression is used as a baseline model. It assumes a linear relationship between AAPL and the other stock prices in the dataset.

This model provides a simple benchmark to evaluate how well more advanced models perform.

In [15]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

y_pred_lr = lr_model.predict(X_test)

lr_mae = mean_absolute_error(y_test, y_pred_lr)
lr_rmse = np.sqrt(mean_squared_error(y_test, y_pred_lr))
lr_r2 = r2_score(y_test, y_pred_lr)

print("Linear Regression Results")
print("MAE:", lr_mae)
print("RMSE:", lr_rmse)
print("R^2:", lr_r2)

Linear Regression Results
MAE: 0.009949121806772685
RMSE: 0.013880181706046091
R^2: 0.3387935971727821


The Linear Regression model shows a significant improvement after transforming stock prices into returns. The R² value indicates that the model is able to explain a portion of the variation in AAPL returns using the other stock returns.

The relatively low MAE and RMSE values suggest that prediction errors are small in magnitude. However, the moderate R² value reflects the inherent variability and noise present in financial return data, which makes precise prediction more challenging.

Using raw stock prices initially resulted in poor model performance due to non-stationarity and trending behavior. Converting prices to returns significantly improved the model's ability to learn meaningful relationships.

Ridge Regression:
Ridge Regression extends Linear Regression by adding a regularization term to reduce overfitting. This helps improve model stability when features are correlated.

Since Ridge Regression is sensitive to feature scale, StandardScaler is applied to normalize the data before training.

In [16]:
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

ridge_model = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
ridge_model.fit(X_train, y_train)

y_pred_ridge = ridge_model.predict(X_test)

ridge_mae = mean_absolute_error(y_test, y_pred_ridge)
ridge_rmse = np.sqrt(mean_squared_error(y_test, y_pred_ridge))
ridge_r2 = r2_score(y_test, y_pred_ridge)

print("Ridge Regression Results")
print("MAE:", ridge_mae)
print("RMSE:", ridge_rmse)
print("R^2:", ridge_r2)

Ridge Regression Results
MAE: 0.009948514459534838
RMSE: 0.01387969901777657
R^2: 0.33883958375015477


The Ridge Regression model produced results very similar to Linear Regression, with only a slight improvement in performance metrics. The R² value remained nearly the same, indicating that regularization did not significantly enhance the model's ability to explain variation in AAPL returns.

This suggests that overfitting was not a major issue in the Linear Regression model, and that the relationships between features are relatively stable. While Ridge Regression provides additional robustness, its impact on this dataset is minimal.

K-Nearest Neighbors Regression:
K-Nearest Neighbors (KNN) Regression predicts values based on the average of nearby data points in the feature space. Unlike linear models, KNN can capture nonlinear relationships.

Because KNN relies on distance calculations, feature scaling is essential to ensure that all variables contribute equally.

In [17]:
from sklearn.neighbors import KNeighborsRegressor

knn_model = make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=5))
knn_model.fit(X_train, y_train)

y_pred_knn = knn_model.predict(X_test)

knn_mae = mean_absolute_error(y_test, y_pred_knn)
knn_rmse = np.sqrt(mean_squared_error(y_test, y_pred_knn))
knn_r2 = r2_score(y_test, y_pred_knn)

print("KNN Regression Results")
print("MAE:", knn_mae)
print("RMSE:", knn_rmse)
print("R^2:", knn_r2)

KNN Regression Results
MAE: 0.010625591620464047
RMSE: 0.014621294001055218
R^2: 0.26630025668377344


The KNN Regression model produced lower performance compared to the linear models, as indicated by the lower R² value and slightly higher error metrics.

This suggests that nonlinear relationships do not significantly improve prediction accuracy for this dataset. Instead, the relationships between AAPL and the other stock returns appear to be primarily linear.

Additionally, the performance of KNN highlights its sensitivity to noise in financial return data, which can reduce its effectiveness compared to more stable linear models.

Effect of Preprocessing:
To evaluate the importance of preprocessing, models are tested both with and without feature scaling. Scaling is expected to have a significant impact on distance-based models such as KNN.

In [18]:
# Ridge without scaling
ridge_no_scale = Ridge(alpha=1.0)
ridge_no_scale.fit(X_train, y_train)
y_pred_ridge_ns = ridge_no_scale.predict(X_test)

print("Ridge without scaling R2:", r2_score(y_test, y_pred_ridge_ns))
print("Ridge with scaling R2:", ridge_r2)


# KNN without scaling
knn_no_scale = KNeighborsRegressor(n_neighbors=5)
knn_no_scale.fit(X_train, y_train)
y_pred_knn_ns = knn_no_scale.predict(X_test)

print("KNN without scaling R2:", r2_score(y_test, y_pred_knn_ns))
print("KNN with scaling R2:", knn_r2)

Ridge without scaling R2: 0.32311202534147476
Ridge with scaling R2: 0.33883958375015477
KNN without scaling R2: 0.2768504979528531
KNN with scaling R2: 0.26630025668377344


Feature scaling had a modest impact on model performance. Ridge Regression showed a slight improvement when scaling was applied, which is expected since regularization can be influenced by feature magnitude.

For KNN, scaling did not significantly improve performance and slightly decreased the R² value. This is likely because the dataset was already transformed into returns, which are naturally on a similar scale across features.

Overall, these results suggest that preprocessing is important, but its impact depends on both the model and the nature of the data.

In [19]:
results = pd.DataFrame({
    "Model": ["Linear Regression", "Ridge Regression", "KNN Regression"],
    "MAE": [lr_mae, ridge_mae, knn_mae],
    "RMSE": [lr_rmse, ridge_rmse, knn_rmse],
    "R2": [lr_r2, ridge_r2, knn_r2]
})

results.sort_values(by="R2", ascending=False)

,Model,MAE,RMSE,R2
1,Ridge Regression,0.009949,0.013880,0.338840
0,Linear Regression,0.009949,0.013880,0.338794
2,KNN Regression,0.010626,0.014621,0.266300


Final Model Comparison:
Among the models tested, Linear Regression and Ridge Regression provided the best performance, with nearly identical R² values. This indicates that the relationships between AAPL and the other stock returns are primarily linear.

KNN Regression performed worse, suggesting that nonlinear methods do not offer significant advantages for this dataset. Additionally, the variability of financial returns likely limits the effectiveness of distance-based models.

Overall, the results demonstrate that simpler linear models are sufficient for capturing relationships in stock return data, while also being more stable and interpretable.